# Draw Keyjoint

## PyTorch

In [1]:
import torch
import csv
import json
import os
import pandas as pd
import numpy as np
import torchvision
from torchvision.utils import draw_keypoints, make_grid, draw_bounding_boxes
from PIL import Image
import torchvision.transforms.functional as F
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = torchvision.models.detection.keypointrcnn_resnet50_fpn(pretrained=True)
model.to(device)
model.eval()

c:\Users\denni\OneDrive\Documents\Keypoint_R-CNN\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\denni\OneDrive\Documents\Keypoint_R-CNN\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=KeypointRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=KeypointRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


KeypointRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(640, 672, 704, 736, 768, 800), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.

In [3]:
# refer online
COCO_SKELETON_0_INDEXED = [
    (1, 0), (2, 0),
    (3, 1), (4, 2),
    (5, 6),
    (5, 7), (6, 8),
    (7, 9), (8, 10),
    (5, 11), (6, 12),
    (11, 13), (12, 14),
    (13, 15), (14, 16)
]
parts = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]
column_names = []
for part in parts:
    column_names.extend([f"{part}_x", f"{part}_y", f"{part}_v"])
keypoint_data = []

In [4]:
data_path = Path('Sitting Posture v2.v4i.coco/train/_annotations.coco.json')

with open(data_path, 'r') as f:
    coco_data = json.load(f)

In [5]:
def get_keypoints(image_id):
  #COCO JSON
  img_info = next(item for item in coco_data['images'] if item['id'] == image_id)
  file_name = img_info['file_name']

  ann_info = [item for item in coco_data['annotations'] if item['image_id'] == image_id]

  image_path = os.path.join('Sitting Posture v2.v4i.coco/train/', file_name)

  img = Image.open(image_path).convert("RGB")
  img_tensor = torch.as_tensor(np.array(img)).permute(2, 0, 1)

  bbox = []
  cat_id = 0
  for ann in ann_info:
    x,y,w,h = ann['bbox']
    cat_id = ann['category_id']
    bbox.append([x,y,x+w,y+h])

  bbox_tensor = torch.tensor(bbox, dtype=torch.float32)
  img_with_boxes = draw_bounding_boxes(img_tensor, bbox_tensor, colors='red',width=2)
  img_for_drawing = F.to_tensor(img).unsqueeze(0).to(device)

  with torch.no_grad():
      output = model(img_for_drawing)[0]

  keypoints = output['keypoints']
  keypoints_scores = output['keypoints_scores']

  #Skip if empty result
  if (keypoints.shape[0] == 0) : return
  scores = output['scores']

  # 90% con7firm result
  high_score_kp = keypoints[scores > 0.9]
  high_score_kp_scores = keypoints_scores[scores > 0.9]
  #debug
  print(scores)
  print(high_score_kp)

  if high_score_kp.shape[0] == 0: return

  #if more than one
  selected_kp = high_score_kp[0]
  selected_kp_scores = high_score_kp_scores[0]
  print(selected_kp_scores)

  kp_filtered = selected_kp.clone()

  for i in range(17):
    if selected_kp_scores[i] < 0.0:
        kp_filtered[i, :] = torch.tensor([0.0, 0.0, 0.0], device=kp_filtered.device)

  #store csv
  data_row = kp_filtered[:, :3].reshape(-1).cpu().numpy()
  df = pd.DataFrame([data_row], columns=column_names)
  df.insert(0, 'image_id', image_id)
  df['cat_id'] = cat_id
  keypoint_data.append(df)

  csv_filename = "keypoint_train_data.csv"
  file_exists = os.path.isfile(csv_filename)

  # Combine the data into a single standard Python list
  full_row = [image_id, cat_id] + data_row.tolist()

  # Open the file and write directly
  with open(csv_filename, mode='a', newline='') as f:
    writer = csv.writer(f)
    
    # Write the header only if the file was just created
    if not file_exists:
      writer.writerow(['image_id', 'cat_id'] + column_names) 
        
    writer.writerow(full_row)
    
  #draw kp on image

  if len(selected_kp[0]) > 0:
    pred_keypoints, visibility = kp_filtered.split([2, 1], dim=-1)
    visibility = visibility.bool()

    annotated_img = draw_keypoints(
        img_with_boxes,
        pred_keypoints.unsqueeze(0),
        visibility=visibility.unsqueeze(0),
        colors='red',
        radius=4,
        connectivity=COCO_SKELETON_0_INDEXED
    )

  #just tengok
  grid = make_grid(annotated_img)
  grid_np = grid.permute(1, 2, 0).numpy() #(H,W,C)

  plt.figure(figsize=(10, 10))
  plt.imshow(grid_np)
  plt.title("Torch!")
  plt.axis('off')
  plt.show()

  output_filename = "keyjoint_train_dataset/keyjoints_" + os.path.basename(image_path)
  torchvision.io.write_png((annotated_img).to(torch.uint8), output_filename)

In [6]:
import os

# Create the output directory if it doesn't exist
output_dir = "keyjoint_train_dataset"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

for image_info in coco_data['images']:
    image_id = image_info['id']
    print(f"Processing image ID: {image_id}")
    get_keypoints(image_id=image_id)

Processing image ID: 0


KeyboardInterrupt: 

In [ ]:
get_keypoints(image_id=2)

In [ ]:
combined_keypoint_df = pd.concat(keypoint_data, ignore_index=True)
combined_keypoint_df.to_csv("keypoint_train_data.csv", index=False)

In [ ]:
!zip -r /content/keyjoint_train_dataset.zip /content/keyjoint_train_dataset